# Chapter 05-07 · Capacity: polynomial features, interactions, under- and overfitting

**Label:** Core  |  **Time:** ~55 minutes  |  **Difficulty:** the vocabulary everything after this
depends on

**Prerequisites:** 05-03 for multiple regression, 05-04 for R-squared on held-out rows, 05-05 for
residual plots, 04-03 for splitting.

**Position in the learning path:** module 05, chapter 7 of 12.

---

## Why this matters

05-05 found a curve in the residuals and fixed it by adding `stops_squared`. That worked, and it raises
the obvious question: **why not add every squared term, and every cubed term, and every product?**

05-04 already showed where that goes - a degree-18 polynomial scoring R-squared 0.80 on its fitting rows
and **-1850** on fresh ones. So there is a limit. This chapter is about where it is, what the two
failures on either side of it look like, and the one kind of extra term that is almost always worth
adding.

**Capacity** is the word for how much a model can express. Too little and it cannot represent the truth -
**underfitting**. Too much and it represents the noise as well - **overfitting**. Both look like a bad
number and they need opposite fixes, so telling them apart is the skill.

## What you will be able to do

- Say what capacity is and what controls it in a linear model
- Add polynomial and interaction terms deliberately, and read the resulting coefficients
- Recognise an interaction from a table of group means, before fitting anything
- Tell underfitting from overfitting using two numbers and a residual plot
- Explain why the same model can be reckless on 30 rows and harmless on 3,000
- Expand features without destroying the numerical conditioning of the problem

## Warm-up: retrieve, do not reread

1. In 05-04, what did R-squared do on the fitting rows as the polynomial degree rose, and on fresh rows?
2. In 05-05, what does a ∪ shape in a residual plot mean?
3. In 05-06, what happened to the condition number when the features were standardised?

<br>

*Answers: (1) it rose to about 0.83 and stayed; on fresh rows it fell to -1850.87. (2) a relationship is
not straight - the model is missing a shape. (3) 365,727 became 1.09.*

## Capacity, and the thing that controls it

**Capacity is the size of the set of functions a model can produce.** A straight line through two-dimensional
data can be any line and nothing else. Add a squared term and it can be any parabola, which includes every
line, so capacity has strictly grown.

For a linear model the count of columns is a workable proxy: **more columns, more capacity**. And columns
multiply faster than anyone expects.

In [ ]:
from math import comb

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import PolynomialFeatures, StandardScaler

table = []
for features in [1, 2, 5, 10, 20]:
    table.append({"original features": features,
                  **{"degree %d" % degree: comb(features + degree, degree) - 1
                     for degree in [1, 2, 3, 5]}})
print("columns produced by a full polynomial expansion")
print(pd.DataFrame(table).to_string(index=False))

check = PolynomialFeatures(3, include_bias=False).fit_transform(np.ones((2, 5)))
print("\nsklearn agrees: 5 features to degree 3 gives %d columns" % check.shape[1])

**Twenty features expanded to degree 3 is 1,770 columns; to degree 5 it is 53,129.**

The formula is `C(p + d, d) - 1`, and the growth is the first practical constraint: a full expansion is
usually not something you can afford, and if you could afford it you could not fit it - 53,129 columns
on any realistic number of rows is a model with more parameters than data.

**So capacity is not something you turn up. It is something you spend**, and the rest of this chapter is
about spending it where it pays.

## The one expansion that usually pays: interactions

A `LinearRegression` says each feature contributes a fixed amount, *whatever the other features are
doing*. That is 05-03's "holding the others constant", and it is often false.

Here is a shop. Sales depend on footfall, on whether there is a promotion, and on whether it is the
weekend - and **the promotion works far better at the weekend**, because that is when people have time to
act on it.

In [ ]:
# SYNTHETIC: 600 shop-days.
# TRUTH: sales = 200 + 0.9 x footfall + 15 x promo + 40 x weekend
#                    + 120 x (promo AND weekend) + noise(sd 25)
shop_rng = np.random.default_rng(9)
n_days = 600
footfall = shop_rng.uniform(50, 400, n_days)
promo = shop_rng.integers(0, 2, n_days).astype(float)
weekend = shop_rng.integers(0, 2, n_days).astype(float)
sales = (200 + 0.9 * footfall + 15 * promo + 40 * weekend
         + 120 * promo * weekend + shop_rng.normal(0, 25, n_days))

shop = pd.DataFrame({"footfall": footfall, "promo": promo, "weekend": weekend})

# subtract the footfall effect so the four cells are comparable
adjusted = pd.DataFrame({"promo": promo, "weekend": weekend,
                         "sales less footfall": sales - 0.9 * footfall})
cells = adjusted.groupby(["weekend", "promo"])["sales less footfall"].mean().unstack()
cells.index = ["weekday", "weekend"]
cells.columns = ["no promo", "promo"]
print(cells.round(1).to_string())
print("\nthe promotion is worth %.1f on a weekday and %.1f at the weekend"
      % (cells.loc["weekday", "promo"] - cells.loc["weekday", "no promo"],
         cells.loc["weekend", "promo"] - cells.loc["weekend", "no promo"]))

> **The promotion is worth 21.1 on a weekday and 139.6 at the weekend.**
>
> **That is what an interaction is**, and you can see it in a two-by-two table of means without fitting
> anything. When the effect of one variable depends on the value of another, no single coefficient can
> describe it.

### Predict before running

A model given `footfall`, `promo` and `weekend` - but not their product - has to report **one** number for
the promotion's effect. What number will it choose, and what will its R-squared be?

In [ ]:
train_shop, test_shop, train_sales, test_sales = train_test_split(
    shop, sales, test_size=0.3, random_state=0)

main_effects = LinearRegression().fit(train_shop, train_sales)
main_prediction = main_effects.predict(test_shop)

with_product = train_shop.assign(promo_x_weekend=train_shop.promo * train_shop.weekend)
test_product = test_shop.assign(promo_x_weekend=test_shop.promo * test_shop.weekend)
interacted = LinearRegression().fit(with_product, train_sales)
interacted_prediction = interacted.predict(test_product)

comparison = pd.DataFrame([
    {"model": "main effects only", "columns": train_shop.shape[1],
     "test R2": r2_score(test_sales, main_prediction),
     "test RMSE": np.sqrt(((test_sales - main_prediction) ** 2).mean())},
    {"model": "with promo x weekend", "columns": with_product.shape[1],
     "test R2": r2_score(test_sales, interacted_prediction),
     "test RMSE": np.sqrt(((test_sales - interacted_prediction) ** 2).mean())},
])
print(comparison.round(4).to_string(index=False))
print("\nthe noise the data was built with had sd 25\n")

print("main effects only  :", dict(zip(train_shop.columns, np.round(main_effects.coef_, 2))))
print("with the product   :", dict(zip(with_product.columns, np.round(interacted.coef_, 2))))
print("the truth          : {'footfall': 0.9, 'promo': 15, 'weekend': 40, 'promo_x_weekend': 120}")

**The main-effects model reports the promotion is worth +80.76 - a number that is correct for nobody.**

The truth is 15 on a weekday and 135 at the weekend, and the model, forced to pick one, has landed
roughly in the middle. Every decision made from that coefficient is wrong: promotions look
disappointing at the weekend and far too good midweek.

**And its R-squared is 0.9074.** This is the point worth carrying out of the section - a model can score
0.91 on held-out data and still be wrong about the only question anyone asked it.

Adding one column of `promo * weekend` takes R-squared to **0.9630** and RMSE to **23.11**, against noise
built with standard deviation 25. The model is now as good as the data allows, and its coefficients -
0.91, 20.63, 41.77, 120.02 - recover the truth of 0.9, 15, 40, 120.

**How to read an interaction coefficient:** `promo` is now the effect of a promotion **when
`weekend` is 0**, and `promo_x_weekend` is the *extra* effect when it is 1. The weekend total is
20.63 + 120.02 = 140.65. Main-effect coefficients stop being general statements once an interaction is
present, and reading them as though they were is a common and expensive mistake.

In [ ]:
fig, (left, right) = plt.subplots(1, 2, figsize=(12.8, 4.6))

positions = np.arange(2)
width = 0.36
left.bar(positions - width / 2, cells["no promo"].to_numpy(), width, color="#999999",
         label="no promo")
left.bar(positions + width / 2, cells["promo"].to_numpy(), width, color="#D55E00",
         label="promo")
for index, day in enumerate(cells.index):
    lift = cells.loc[day, "promo"] - cells.loc[day, "no promo"]
    left.annotate("", xy=(index + width / 2, cells.loc[day, "promo"]),
                  xytext=(index + width / 2, cells.loc[day, "no promo"]),
                  arrowprops=dict(arrowstyle="<->", color="#000000", linewidth=1.6))
    left.text(index + width / 2 + 0.06, (cells.loc[day, "promo"]
                                         + cells.loc[day, "no promo"]) / 2,
              "+%.0f" % lift, fontsize=11, fontweight="bold")
left.set_xticks(positions)
left.set_xticklabels(cells.index)
left.set_ylabel("sales, footfall effect removed")
left.set_title("The two arrows are different lengths.\nThat is the interaction.", fontsize=11)
left.legend(fontsize=9)

for weekend_value, colour, marker in [(0, "#0072B2", "o"), (1, "#D55E00", "s")]:
    means = [cells.iloc[weekend_value, 0], cells.iloc[weekend_value, 1]]
    right.plot([0, 1], means, marker + "-", color=colour, linewidth=2.4, markersize=10,
               label=cells.index[weekend_value])
right.set_xticks([0, 1])
right.set_xticklabels(["no promo", "promo"])
right.set_ylabel("sales, footfall effect removed")
right.set_title("The same thing as a slope: non-parallel lines\nmean an interaction",
                fontsize=11)
right.legend(fontsize=9)

plt.tight_layout()
plt.show()

**The right-hand panel is the diagnostic worth memorising.** Plot the group means with one line per
level of the second variable:

- **Parallel lines: no interaction.** The effect of the promotion is the same whichever day it is, so one
  coefficient describes it.
- **Non-parallel lines: an interaction.** The steeper the divergence, the more the product term is worth.
- **Lines that cross:** the effect changes *sign*, and a main-effects model will report something near
  zero and conclude the variable does not matter.

**This costs one `groupby` and no model.** For a categorical pair it is the fastest check in this
chapter, and for two continuous features the equivalent is to bin one of them into thirds and plot the
same picture.

### Which interactions to add

The combinatorial table is why "add them all" fails: 20 features have 190 pairwise products, before any
squared terms. Three approaches, in the order they are worth trying:

1. **Domain knowledge.** Someone who works with the data knows that promotions land differently at the
   weekend. This is by far the best source and it is free.
2. **The non-parallel-lines plot**, on the handful of pairs you have a reason to suspect.
3. **Let a model that finds them automatically tell you.** Trees and gradient boosting (05-10, 05-11)
   represent interactions natively without being asked; if boosting substantially beats your linear
   model, an unmodelled interaction is one of the likely reasons.

## Polynomial terms, and the shape of the trade-off

Interactions were nearly free - one column, a large gain, and the coefficient still readable. Polynomial
degree is the opposite: **every extra degree adds capacity you may not need, and you will not find out on
the training rows.**

Here is a curve with a known truth - a cubic - and 60 rows to learn it from.

In [ ]:
# SYNTHETIC: 120 points on a cubic. TRUTH: 2 + 1.5x - 0.8x^2 + 0.15x^3, noise sd 3.0
curve_rng = np.random.default_rng(4)
n_points = 120
position = curve_rng.uniform(-4, 4, n_points)
outcome = (2.0 + 1.5 * position - 0.8 * position ** 2 + 0.15 * position ** 3
           + curve_rng.normal(0, 3.0, n_points))

fit_x, held_x, fit_y, held_y = train_test_split(
    position, outcome, test_size=0.5, random_state=0)
print("%d rows to fit, %d held out, noise sd 3.0\n" % (len(fit_x), len(held_x)))


def polynomial_fit(degree, train_x, train_y):
    model = make_pipeline(PolynomialFeatures(degree, include_bias=False),
                          StandardScaler(), LinearRegression())
    return model.fit(train_x.reshape(-1, 1), train_y)


def rmse(model, x, y):
    return float(np.sqrt(((y - model.predict(x.reshape(-1, 1))) ** 2).mean()))


degrees = list(range(1, 19))
curve_table = []
for degree in degrees:
    model = polynomial_fit(degree, fit_x, fit_y)
    curve_table.append({"degree": degree,
                        "train RMSE": rmse(model, fit_x, fit_y),
                        "held-out RMSE": rmse(model, held_x, held_y)})
curve_table = pd.DataFrame(curve_table)
print(curve_table[curve_table.degree.isin([1, 2, 3, 5, 8, 12, 16, 18])]
      .round(4).to_string(index=False))

winner = int(curve_table.loc[curve_table["held-out RMSE"].idxmin(), "degree"])
print("\nlowest held-out RMSE at degree %d (%.4f); the data was generated at degree 3"
      % (winner, curve_table["held-out RMSE"].min()))

**The held-out column finds degree 3 - the degree the data was actually built at - and the training
column does not.**

Read the two columns side by side, because they say different things at every point:

| | train RMSE | held-out RMSE | what it is |
|---|---|---|---|
| degree 1 | 5.6098 | **5.1607** | underfitting |
| degree 3 | 2.7855 | **3.2602** | about right |
| degree 18 | 2.4392 | **3.8617** | overfitting |

**Degree 1 is the interesting row.** Its held-out RMSE is *better* than its training RMSE, which looks
impossible and is not - a model too simple to fit the noise cannot fit the training noise either, so
there is no gap for it to lose. **A held-out score at or above the training score is a signature of
underfitting**, and it is the one case where the absence of a gap is bad news.

From degree 3 onwards the training RMSE keeps falling - 2.79 to 2.44 - and every bit of that improvement
is noise being memorised, because the truth stopped changing at degree 3.

In [ ]:
fig, (left, right) = plt.subplots(1, 2, figsize=(13, 4.6))

left.plot(curve_table.degree, curve_table["train RMSE"], "o-", color="#0072B2",
          linewidth=2.2, markersize=6, label="on the 60 fitting rows")
left.plot(curve_table.degree, curve_table["held-out RMSE"], "s-", color="#D55E00",
          linewidth=2.2, markersize=6, label="on the 60 held-out rows")
left.axhline(3.0, color="#000000", linestyle="--", linewidth=1.5,
             label="the noise floor, sd 3.0")
left.axvline(winner, color="#009E73", linewidth=2, alpha=0.6)
left.text(winner + 0.35, 5.3, "best held-out\ndegree %d" % winner, color="#009E73", fontsize=9.5,
          fontweight="bold")
left.set_xlabel("polynomial degree")
left.set_ylabel("RMSE")
left.set_xticks([1, 3, 6, 9, 12, 15, 18])
left.set_title("Training error only ever falls. Held-out error turns.", fontsize=11)
left.legend(fontsize=8.5)

grid = np.linspace(-4.2, 4.2, 400)
right.scatter(fit_x, fit_y, s=22, color="#666666", alpha=0.65, label="the 60 fitting rows")
right.plot(grid, 2.0 + 1.5 * grid - 0.8 * grid ** 2 + 0.15 * grid ** 3,
           color="#000000", linewidth=2.4, linestyle="--", label="the truth")
for degree, colour in [(1, "#0072B2"), (3, "#009E73"), (18, "#D55E00")]:
    model = polynomial_fit(degree, fit_x, fit_y)
    right.plot(grid, model.predict(grid.reshape(-1, 1)), color=colour, linewidth=2.2,
               label="degree %d" % degree)
right.set_ylim(fit_y.min() - 8, fit_y.max() + 8)
right.set_xlabel("x")
right.set_ylabel("y")
right.set_title("Too stiff, about right, and chasing noise", fontsize=11)
right.legend(fontsize=8.5)

plt.tight_layout()
plt.show()

**The right-hand panel is what the numbers mean.**

- **Degree 1** cannot bend at all. It misses the shape everywhere, and it would miss it just as badly with
  a million rows - **underfitting is not fixed by more data.**
- **Degree 3** sits on the truth almost exactly.
- **Degree 18** passes closer to individual points and wanders between them, most violently near the
  edges where it has the fewest rows to constrain it. Every wiggle is a feature of *these* 60 rows.

**The two failures have opposite fixes.** Underfitting needs more capacity - a higher degree, an
interaction, a different model. Overfitting needs less - a lower degree, fewer columns, a penalty
(05-09), or more rows. **Applying either fix to the other problem makes it worse**, which is why "the
score is bad" is never an actionable statement on its own.

## Underfitting and overfitting, told apart in practice

Three signals, in the order they are worth checking.

In [ ]:
diagnostic = []
for degree in [1, 3, 18]:
    model = polynomial_fit(degree, fit_x, fit_y)
    train_error = rmse(model, fit_x, fit_y)
    held_error = rmse(model, held_x, held_y)
    diagnostic.append({"degree": degree, "train RMSE": train_error,
                       "held-out RMSE": held_error,
                       "gap": held_error - train_error,
                       "train RMSE vs noise floor": train_error - 3.0})
print(pd.DataFrame(diagnostic).round(4).to_string(index=False))

**1. The gap between training and held-out error.** Small gap plus high error means underfitting; large
gap means overfitting. Degree 1 has a gap of **-0.45** and degree 18 has **+1.42**.

**2. The training error against the noise floor.** A training error far *above* what the data's own noise
allows means the model cannot represent the truth, whatever the gap says. Degree 1 sits 2.61 above the
floor of 3.0; degree 3 is 0.21 *below* it, which is the mild optimism of scoring on fitting rows.

**3. The residual plot of 05-05.** Underfitting leaves *structure* - a curve, a fan, a systematic sign -
because the missing part of the truth is still in the residuals. Overfitting leaves a training residual
plot that looks perfect and a held-out one that does not.

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(14, 7), sharex=True, sharey=True)

for column, degree in enumerate([1, 3, 18]):
    model = polynomial_fit(degree, fit_x, fit_y)
    for row, (x_values, y_values, label, colour) in enumerate([
            (fit_x, fit_y, "fitting rows", "#0072B2"),
            (held_x, held_y, "held-out rows", "#D55E00")]):
        residual = y_values - model.predict(x_values.reshape(-1, 1))
        axes[row, column].scatter(x_values, residual, s=26, alpha=0.7, color=colour)
        axes[row, column].axhline(0, color="#000000", linewidth=1.4)
        axes[row, column].set_title("degree %d, %s\nRMSE %.2f"
                                    % (degree, label, np.sqrt((residual ** 2).mean())),
                                    fontsize=10)
        if column == 0:
            axes[row, column].set_ylabel("residual")
        if row == 1:
            axes[row, column].set_xlabel("x")

axes[0, 0].set_ylim(-13, 13)
plt.tight_layout()
plt.show()

**Read the top row left to right, then the bottom row.**

- **Degree 1, both rows: a clear ∩ shape.** The missing cubic is sitting in the residuals in plain sight.
  This is 05-05's curvature signature, and it appears on *both* the fitting and the held-out rows -
  which is the tell for underfitting. The fault is in the model, so it follows the model everywhere.
- **Degree 3, both rows: a random cloud.** Nothing left to extract.
- **Degree 18: the top row is the tightest cloud of the three and the bottom row is the worst.** The
  fault appears only where the model has not seen the data, which is the tell for overfitting.

> **Underfitting shows up on the training rows. Overfitting is invisible there by construction.**
>
> That single sentence is why 04-03's held-out split exists, and why a residual plot drawn only on
> training rows can certify a model that is about to fail.

## Capacity is only meaningful relative to the number of rows

"Degree 18 overfits" is not a fact about degree 18. Here is the same sweep at three sample sizes.

### Predict before running

Degree 16 was a disaster on 60 fitting rows. What will it do on 1,800?

In [ ]:
def sweep_at(total_rows, seed=4):
    local = np.random.default_rng(seed)
    x_here = local.uniform(-4, 4, total_rows)
    y_here = (2.0 + 1.5 * x_here - 0.8 * x_here ** 2 + 0.15 * x_here ** 3
              + local.normal(0, 3.0, total_rows))
    train_x, test_x, train_y, test_y = train_test_split(
        x_here, y_here, test_size=0.4, random_state=0)
    out = {}
    for degree in [1, 2, 3, 5, 8, 12, 16]:
        model = polynomial_fit(degree, train_x, train_y)
        out[degree] = r2_score(test_y, model.predict(test_x.reshape(-1, 1)))
    return len(train_x), out


rows = []
for total in [30, 300, 3000]:
    train_count, scores = sweep_at(total)
    rows.append({"rows used to fit": train_count,
                 **{"degree %d" % d: v for d, v in scores.items()}})
print("held-out R-squared")
print(pd.DataFrame(rows).round(3).to_string(index=False))

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4.6))

for total, colour, marker in [(30, "#D55E00", "s"), (300, "#0072B2", "o"),
                              (3000, "#009E73", "^")]:
    train_count, scores = sweep_at(total)
    ax.plot(list(scores.keys()), list(scores.values()), marker + "-", color=colour,
            linewidth=2.2, markersize=8, label="%d rows to fit" % train_count)

ax.axhline(0, color="#000000", linewidth=1.4)
ax.set_ylim(-3.4, 1.05)
ax.set_xlabel("polynomial degree")
ax.set_ylabel("held-out R-squared")
ax.set_title("The same degree 16 is ruinous on 18 rows and free on 1,800", fontsize=11.5)
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

**Degree 16 scores -3.107 on 18 fitting rows and +0.874 on 1,800.**

The model class did not change. The data did.

- **18 rows:** every degree above 5 collapses, and the best score available is 0.145. There are not enough
  rows to pin down that many coefficients, so they get set by noise.
- **180 rows:** the peak is 0.868 at degree 5, and degree 16 still manages 0.863. The penalty for excess
  capacity has almost vanished.
- **1,800 rows:** degree 3 wins at 0.875 and degree 16 is 0.874 - **a difference of one thousandth.**
  Thirteen unnecessary parameters cost essentially nothing.

**Which yields the rule of thumb this chapter exists to justify:**

> **Overfitting is a statement about the ratio of parameters to rows, not about the model.** The same
> expansion is reckless in one dataset and free in another, so a fixed rule like "never go above degree 3"
> is wrong in both directions.
>
> And the corollary: **more data is a real fix for overfitting, and no fix at all for underfitting.**
> Degree 1 scored -0.703, +0.681 and +0.634 at the three sizes - it never improves past its own ceiling.

**How to actually choose the degree:** you do not reason about it, you measure it. That is what 04-07's
cross-validation is for, and 05-09's regularisation offers the alternative of keeping the capacity and
constraining it instead.

## Failure lab: expanding features the naive way

05-06 left a promise. Raw powers of `x` are a numerically terrible basis, and here is what that costs.

In [ ]:
def condition_of(matrix):
    with_intercept = np.column_stack([np.ones(len(matrix)), matrix])
    return float(np.linalg.cond(with_intercept.T @ with_intercept / len(matrix)))


conditioning = []
for degree in [3, 8, 12]:
    raw_powers = PolynomialFeatures(degree, include_bias=False).fit_transform(
        fit_x.reshape(-1, 1))
    scaled = StandardScaler().fit_transform(raw_powers)
    conditioning.append({"degree": degree,
                         "raw powers": condition_of(raw_powers),
                         "standardised after expanding": condition_of(scaled)})
print(pd.DataFrame(conditioning).to_string(index=False,
                                           float_format=lambda v: "%.4g" % v))

**At degree 12 the raw expansion has a condition number of 7.5e+14.**

Double precision carries about 16 significant digits, so a condition number of 1e15 means the fitted
coefficients have **roughly one digit of meaning left**. The matrix is not quite singular; it is close
enough that the solver's answer is mostly rounding error.

The cause is scale. `fit_x` runs to about 4, so the degree-12 column runs to `4^12` - about 17 million -
next to a column of ones. **That is 05-06's ravine, built deliberately by the feature expansion.**

**Three fixes, in ascending order of quality:**

1. **Standardise after expanding.** One line, and it takes degree 12 from 7.5e+14 to 1.4e+08 - a
   ten-million-fold improvement, and the reason `StandardScaler` sits inside every pipeline in this
   chapter.
2. **Centre before expanding.** Powers of `x - mean(x)` are far better behaved than powers of `x`,
   because the values no longer all share a sign.
3. **Use an orthogonal basis** - Legendre or Chebyshev polynomials, via `numpy.polynomial`. These are
   constructed so the columns are uncorrelated by design, and the conditioning problem does not arise.

**And the practical point:** the middle column of that table is why 05-04's degree-18 R-squared went
*down* between degree 12 and 18 when the theory says it cannot. It was not a statistical effect. It was
the arithmetic running out of digits.

## The whole picture on one page

In [ ]:
fig, ax = plt.subplots(figsize=(12.5, 5.6))
ax.set_xlim(0, 10.6)
ax.set_ylim(0, 6.8)
ax.axis("off")

ax.add_patch(plt.Rectangle((0.2, 3.6), 3.2, 2.6, facecolor="#DDEBF7",
                           edgecolor="#0072B2", linewidth=1.6))
ax.text(1.8, 5.85, "UNDERFITTING", fontsize=12, fontweight="bold", ha="center")
for offset, line in enumerate([
        "training error is high",
        "gap to held-out is small",
        "  (or held-out looks better)",
        "residuals show STRUCTURE",
        "  on training rows too",
        "more data does not help"]):
    ax.text(0.35, 5.45 - offset * 0.32, line, fontsize=9.5)

ax.add_patch(plt.Rectangle((3.7, 3.6), 3.2, 2.6, facecolor="#D9EAD3",
                           edgecolor="#009E73", linewidth=1.6))
ax.text(5.3, 5.85, "ABOUT RIGHT", fontsize=12, fontweight="bold", ha="center")
for offset, line in enumerate([
        "training error near the",
        "  noise floor",
        "small, stable gap",
        "residuals are a cloud",
        "  on BOTH sets",
        "found by held-out score"]):
    ax.text(3.85, 5.45 - offset * 0.32, line, fontsize=9.5)

ax.add_patch(plt.Rectangle((7.2, 3.6), 3.2, 2.6, facecolor="#F4CCCC",
                           edgecolor="#D55E00", linewidth=1.6))
ax.text(8.8, 5.85, "OVERFITTING", fontsize=12, fontweight="bold", ha="center")
for offset, line in enumerate([
        "training error very low",
        "gap to held-out is large",
        "training residuals look",
        "  perfect; held-out do not",
        "more data DOES help",
        "so does less capacity"]):
    ax.text(7.35, 5.45 - offset * 0.32, line, fontsize=9.5)

ax.annotate("", xy=(7.1, 4.9), xytext=(3.5, 4.9),
            arrowprops=dict(arrowstyle="<|-|>", linewidth=2.4, color="#666666"))
ax.text(5.3, 3.15, "MORE CAPACITY  ->", fontsize=11, ha="center", fontweight="bold",
        color="#666666")

ax.text(0.2, 2.6, "WHAT ADDS CAPACITY", fontsize=11.5, fontweight="bold")
for offset, (item, note) in enumerate([
        ("interactions  a x b", "one column, often a large gain - check with non-parallel lines"),
        ("polynomial degree", "C(p+d, d) - 1 columns; 20 features at degree 3 is 1,770"),
        ("more features", "each one is capacity, whether or not it helps"),
        ("a flexible model class", "trees, boosting - they find interactions unasked")]):
    ax.text(0.35, 2.15 - offset * 0.45, "* %s" % item, fontsize=10, fontweight="bold")
    ax.text(3.3, 2.15 - offset * 0.45, note, fontsize=9, color="#444444")

plt.tight_layout()
plt.show()

## Common misconceptions

**"Overfitting means the model is too complicated."**
Too complicated **for this many rows**. Degree 16 was ruinous on 18 rows and free on 1,800. The number
that matters is parameters against rows, not parameters.

**"A high R-squared means the model is right."**
The main-effects shop model scored **0.9074 on held-out data** and reported that promotions are worth
+80.76 - a figure that is wrong for weekdays and wrong for weekends. A good score and a wrong answer are
entirely compatible.

**"Add every interaction and let the model sort it out."**
Twenty features have 190 pairwise products before any squared terms, and each is capacity spent. Add the
ones you have a reason for, then check whether a model that finds them automatically beats you.

**"The training residual plot is clean, so the model is fine."**
Overfitting is invisible on training rows by construction - it *is* the model fitting those rows. Degree
18 had the tightest training residuals in this chapter and the worst held-out ones.

**"More data fixes overfitting, so more data is the answer."**
More data fixes overfitting and does nothing for underfitting. Degree 1 scored -0.70, +0.68 and +0.63 on
18, 180 and 1,800 rows: it stops improving, because the fault is that it cannot bend.

**"Polynomial features are just `PolynomialFeatures`."**
Not without scaling them afterwards. Degree 12 raw has a condition number of 7.5e+14, which leaves about
one meaningful digit in the coefficients.

**"The degree that fits best is the true degree."**
The held-out score found degree 3 here because the truth was a cubic and there were enough rows. On 180
rows it preferred degree 5, and on 18 rows the whole exercise is noise. **Held-out selection estimates
what predicts well, not what is true.**

## Exercises

Solutions: `solutions/05_regression/05-07_capacity_solutions.ipynb`.

### Quick understanding

**E1.** Define capacity, and give two things that increase it in a linear model.

**E2.** State the two signatures that distinguish underfitting from overfitting, using training and
held-out error.

**E3.** Why does a residual plot on training rows fail to detect overfitting?

### Hand calculation

**E4.** Group means of `y` are: weekday/no-promo 100, weekday/promo 130, weekend/no-promo 150,
weekend/promo 180. Is there an interaction? Show the arithmetic.

**E5.** Same table, but weekend/promo is 240. Give the four coefficients of a model with an intercept,
`promo`, `weekend` and `promo x weekend`, by hand.

**E6.** How many columns does a full degree-2 expansion of 6 features produce? List the three kinds of
term and count each.

**E7.** A model has training RMSE 2.0 and held-out RMSE 2.1 on data whose irreducible noise has standard
deviation 5.0. Diagnose it, and say what is odd.

### Coding

**E8.** Write `interaction_plot(frame, first, second, target)` that draws the non-parallel-lines
diagnostic for two categorical columns. Run it on the shop data for `promo` against `weekend`, and for
`promo` against a column you create by splitting `footfall` at its median.

**E9.** For the shop data, fit every model from `PolynomialFeatures(degree=1)` to `degree=3` on all three
features and report held-out R-squared and the column count for each. Say which you would ship.

**E10.** Reproduce the capacity curve using 5-fold cross-validation instead of a single split, and report
how much the chosen degree varies across ten different random seeds.

**E11.** Show that centring `x` before expanding improves the conditioning, and quantify by how much at
degrees 3, 8 and 12.

**E12.** Build a dataset where the interaction is the *only* signal - both main effects are exactly
zero - and show what a main-effects model reports for R-squared and for each coefficient.

### Interpretation

**E13.** A colleague reports training R-squared 0.99 and held-out R-squared 0.35, and proposes collecting
more data. Say whether that will work, and what else you would try first.

**E14.** Your model's held-out score is slightly *better* than its training score. Give two explanations
and say how to tell them apart.

### Debugging

**E15.** Adding `PolynomialFeatures(degree=4)` makes held-out performance collapse and produces
coefficients of the order 1e+9. Diagnose it and give the fix.

**E16.** A model with an interaction term reports a `promo` coefficient near zero, and the business
insists promotions work. Explain what may have happened and what to report instead.

### Exam and interview reasoning

**E17.** "How do you know if a model is overfitting?" Answer in under a minute, then handle: "and if you
only have 200 rows and cannot afford a test set?"

### Transfer to a different situation

**E18.** You are predicting hospital readmission from 40 features, 8 of them categorical with many
levels. State how you would think about capacity, what you would expand and what you would not.

### Explain it to someone non-technical

**E19.** In under 90 words, explain overfitting using something other than a curve through points.

### Optional challenge

**E20.** Find, empirically, the number of rows at which degree 16 stops being worse than degree 3 on this
chapter's data generator, by sweeping the sample size. Then explain the shape of the resulting curve.

**E21.** Show that a model with an interaction term and one with the four group means as a categorical
encoding make **identical predictions** on the shop data, and explain why the two parameterisations are
the same model.

## Mastery check

- [ ] Recognise an interaction from a table of group means, before fitting
- [ ] Read a main-effect coefficient correctly when an interaction is present
- [ ] Count the columns a polynomial expansion will produce, before running it
- [ ] Diagnose underfitting and overfitting from two errors and a residual plot
- [ ] Explain why the same degree is reckless on one dataset and free on another
- [ ] Scale after expanding, and say what happens if you do not

## What should now feel instinctive

- Checking for non-parallel lines before adding an interaction, and after a disappointing model
- Reading "training error" and "held-out error" as two different quantities, always both
- Asking "how many rows per parameter?" before asking "which model?"
- Treating structure in a *training* residual plot as underfitting, not noise
- Putting `StandardScaler` after `PolynomialFeatures`, every time

## Flashcards

| Front | Back |
|---|---|
| Capacity | the size of the set of functions a model can express |
| Columns from a full expansion | `C(p + d, d) - 1`; 20 features at degree 3 is 1,770 |
| Interaction | the effect of one variable depends on another's value |
| Spotting one without a model | group means: non-parallel lines |
| Main effect with an interaction present | the effect when the other variable is 0, not in general |
| The shop's main-effects model | R-squared 0.9074 and a promo effect of +80.76, true for nobody |
| Underfitting | high training error, small or negative gap, structure in both residual plots |
| Overfitting | low training error, large gap, training residuals look perfect |
| Degree 16 on 18 rows vs 1,800 | -3.107 against +0.874 |
| More data | fixes overfitting; does nothing for underfitting |
| Raw powers to degree 12 | condition number 7.5e+14 - about one meaningful digit left |

## Next

**05-08 · Bias, variance, and learning curves.** This chapter named the two failures and showed how to
tell them apart. The next one explains *why* they trade off against each other, decomposes the error into
the part caused by a model being too stiff and the part caused by it being too sensitive, and introduces
the learning curve - the plot that answers "would more data help?" before you go and collect it.

The bridge is the sweep you just ran three times at three sample sizes. 05-08 turns that into one plot
you can read in five seconds.